In [1]:
import numpy as np 
import pandas as pd 

In [2]:
sales = pd.read_csv(r"D:\AnalytixLabs\Data Science & AI\6. Case Studies\Machine Learning Case Studies\4. Capstone Case Study - Finding-Marketing-Insights\Datasets\Online_Sales.csv")
customers = pd.read_excel(r"D:\AnalytixLabs\Data Science & AI\6. Case Studies\Machine Learning Case Studies\4. Capstone Case Study - Finding-Marketing-Insights\Datasets\CustomersData.xlsx")
coupon = pd.read_csv(r"D:\AnalytixLabs\Data Science & AI\6. Case Studies\Machine Learning Case Studies\4. Capstone Case Study - Finding-Marketing-Insights\Datasets\Discount_Coupon.csv")
marketing = pd.read_csv(r"D:\AnalytixLabs\Data Science & AI\6. Case Studies\Machine Learning Case Studies\4. Capstone Case Study - Finding-Marketing-Insights\Datasets\Marketing_Spend.csv")
tax = pd.read_excel(r"D:\AnalytixLabs\Data Science & AI\6. Case Studies\Machine Learning Case Studies\4. Capstone Case Study - Finding-Marketing-Insights\Datasets\Tax_amount.xlsx")

In [3]:
sales['Transaction_Date'] = pd.to_datetime(
    sales['Transaction_Date'].astype(str),
    format='%Y%m%d'
)

In [4]:
sales['Month'] = sales['Transaction_Date'].dt.month

In [5]:
sales.columns = sales.columns.str.strip()
coupon.columns = coupon.columns.str.strip()
customers.columns = customers.columns.str.strip()
marketing.columns = marketing.columns.str.strip()
tax.columns = tax.columns.str.strip()

In [6]:
print(sales.columns)
print(coupon.columns)

Index(['CustomerID', 'Transaction_ID', 'Transaction_Date', 'Product_SKU',
       'Product_Description', 'Product_Category', 'Quantity', 'Avg_Price',
       'Delivery_Charges', 'Coupon_Status', 'Month'],
      dtype='object')
Index(['Month', 'Product_Category', 'Coupon_Code', 'Discount_pct'], dtype='object')


In [7]:
print(coupon['Month'].unique())
print(sales['Month'].unique()[:5])

['Jan' 'Feb' 'Mar' 'Apr' 'May' 'Jun' 'Jul' 'Aug' 'Sep' 'Oct' 'Nov' 'Dec']
[1 2 3 4 5]


In [8]:
sales['Month'] = sales['Transaction_Date'].dt.strftime('%b')

In [9]:
sales['Month'].unique()

array(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep',
       'Oct', 'Nov', 'Dec'], dtype=object)

In [10]:
sales = sales.merge(
    coupon,
    on=['Month', 'Product_Category'],
    how='left'
)

In [11]:
sales[['Month','Product_Category','Coupon_Code','Discount_pct']].head()

,Month,Product_Category,Coupon_Code,Discount_pct
0,Jan,Nest-USA,ELEC10,10.0
1,Jan,Nest-USA,ELEC10,10.0
2,Jan,Office,OFF10,10.0
3,Jan,Apparel,SALE10,10.0
4,Jan,Bags,AIO10,10.0


In [14]:
sales['Discount_pct'].isna().sum()

np.int64(0)

In [13]:
sales['Discount_pct'] = sales['Discount_pct'].fillna(0)

In [15]:
sales = sales.merge(
    tax,
    on='Product_Category',
    how='left'
)

In [16]:
sales['Discount_pct'] = sales['Discount_pct'] / 100
sales['GST'] = sales['GST'] / 100

In [17]:
sales['Invoice_Value'] = (
    (
        sales['Quantity'] *
        sales['Avg_Price'] *
        (1 - sales['Discount_pct']) *
        (1 + sales['GST'])
    )
    + sales['Delivery_Charges']
)

In [18]:
sales[['Transaction_ID',
       'Product_SKU',
       'Quantity',
       'Avg_Price',
       'Discount_pct',
       'GST',
       'Delivery_Charges',
       'Invoice_Value']].head()

,Transaction_ID,Product_SKU,Quantity,Avg_Price,Discount_pct,GST,Delivery_Charges,Invoice_Value
0,16679,GGOENEBJ079499,1,153.71,0.1,0.0010,6.5,144.977339
1,16680,GGOENEBJ079499,1,153.71,0.1,0.0010,6.5,144.977339
2,16681,GGOEGFKQ020399,1,2.05,0.1,0.0010,6.5,8.346845
3,16682,GGOEGAAB010516,5,17.53,0.1,0.0018,6.5,85.526993
4,16682,GGOEGBJL013999,1,16.50,0.1,0.0018,6.5,21.376730


In [19]:
total_revenue = sales['Invoice_Value'].sum()
print("Total Revenue:", round(total_revenue, 2))

Total Revenue: 4305601.85
